# 01 — Airbnb Prices in European Cities

**Étape du pipeline** (Ward, Grinstein et Keim) : *Raw data → Data transformation*.

Source : [Kaggle, thedevastator](https://www.kaggle.com/datasets/thedevastator/airbnb-prices-in-european-cities), 20 CSV (10 villes × semaine / week-end). On garde les **8 fichiers des 4 villes communes** avec le dataset hôtels : Londres, Berlin, Paris, Rome.

Sortie : `data/interim/airbnb.parquet`, au schéma commun (`src/prep/schema.py`, `docs/schema.md`).

In [ ]:
import sys
from pathlib import Path

RACINE = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "prep").is_dir())
sys.path.insert(0, str(RACINE))
RAW, INTERIM, PROCESSED = (RACINE / "data" / d for d in ("raw", "interim", "processed"))

import pandas as pd
import matplotlib.pyplot as plt
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)

from src.prep import airbnb, schema
from src.prep.common import METRO_SEUIL_KM

## 1. Chargement des fichiers bruts

In [ ]:
brut = airbnb.charger_brut(RAW / "airbnb")
print(brut.shape)
brut.head()

In [ ]:
brut.dtypes

In [ ]:
brut.groupby(["ville_brut", "periode_brut"]).size().unstack()

**Constat** : aucune valeur manquante, et les noms de colonnes correspondent à ceux du handoff.

In [ ]:
brut.isna().sum().sum()

## 2. Distributions des variables utiles

In [ ]:
cols = ["realSum", "guest_satisfaction_overall", "cleanliness_rating", "dist", "metro_dist", "person_capacity", "bedrooms"]
brut[cols].describe(percentiles=[.01, .5, .99]).round(2)

In [ ]:
brut["room_type"].value_counts()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for v, g in brut.groupby("ville_brut"):
    axes[0].hist(g["realSum"] / 2, bins=80, range=(0, 600), histtype="step", label=v)
    axes[1].hist(g["metro_dist"], bins=60, range=(0, 3), histtype="step", label=v)
axes[0].set(title="Prix par nuit (realSum / 2, €)", xlabel="€")
axes[1].axvline(METRO_SEUIL_KM, color="k", ls="--")
axes[1].set(title="Distance au métro (km)", xlabel="km")
axes[0].legend(); plt.tight_layout()

**Constats**

- `realSum` est le prix total pour 2 personnes et 2 nuits. La queue de distribution est longue : le maximum dépasse 16 000 €. Les aberrants sont écartés à la fusion (notebook 05), par ville et par source.
- `guest_satisfaction_overall` est déjà sur 100.
- La part d'annonces à moins de 500 m du métro est proche de celle des hôtels qui déclarent un accès au métro (comparaison dans le notebook 05).

In [ ]:
(brut["metro_dist"] < METRO_SEUIL_KM).groupby(brut["ville_brut"]).mean().round(2)

## 3. Doublons

In [ ]:
print("lignes strictement identiques :", brut.duplicated().sum())
print("même ville, période, position et prix :", brut.duplicated(["ville_brut", "periode_brut", "lat", "lng", "realSum"]).sum())

**Constat** : aucune ligne n'est strictement identique. Quelques annonces partagent la position et le prix mais diffèrent sur d'autres colonnes, par exemple des logements différents dans un même immeuble. On les garde.

Le dataset ne contient pas d'identifiant d'annonce. On ne peut donc pas savoir si une même annonce figure à la fois en semaine et en week-end : les deux périodes restent des lignes distinctes, repérées par la colonne `periode`.

## 3 bis. Quel « centre » pour la distance ?

Le dataset ne dit pas depuis quel point `dist` est mesurée. On cherche le point qui explique le mieux `dist`, puis on le compare au centre de Booking. Le centre Booking a été estimé par le géocodage des hôtels (handoff du géocodage, G1).

In [ ]:
import numpy as np
from src.prep.common import CENTRES, VILLES, haversine_km

def centre_estime(g, depart, pas=0.02, tours=6):
    lat, lon, d = g["lat"].to_numpy(), g["lng"].to_numpy(), g["dist"].to_numpy()
    c, meilleur = depart, (np.inf, depart)
    for _ in range(tours):  # grille de plus en plus fine autour du meilleur point
        for dla in np.linspace(-5, 5, 21) * pas:
            for dlo in np.linspace(-5, 5, 21) * pas:
                e = np.median(np.abs(haversine_km(lat, lon, c[0] + dla, c[1] + dlo) - d))
                if e < meilleur[0]:
                    meilleur = (e, (c[0] + dla, c[1] + dlo))
        c, pas = meilleur[1], pas / 5
    return meilleur

lignes = []
for cle, (ville, _, _) in VILLES.items():
    g = brut[brut["ville_brut"] == cle].sample(2000, random_state=0)
    ecart, (la, lo) = centre_estime(g, CENTRES[ville])
    lignes.append({"ville": ville, "centre_airbnb": f"{la:.5f}, {lo:.5f}", "ecart_median_m": round(ecart * 1000),
                   "dist_au_centre_booking_km": round(float(haversine_km(la, lo, *CENTRES[ville])), 2)})
pd.DataFrame(lignes)

**Constat** : `dist` est une distance à vol d'oiseau depuis un point fixe, avec un écart médian de quelques mètres. Ce point est le même que celui de Booking à **Londres** (Trafalgar Square) et à **Paris** (Hôtel de Ville). Il diffère à **Berlin**, où c'est Alexanderplatz, à 2,8 km de la porte de Brandebourg, et à **Rome**, où c'est la gare Termini, à 1,75 km de la piazza Venezia.

**Décision (D18)** : on **recalcule** la distance des Airbnb depuis le centre Booking. Les tranches de distance mesurent alors la même chose pour les deux sources. On ne peut pas faire l'inverse : les hôtels sans coordonnées ne sont situés que par leur distance Booking.

## 4. Transformation vers le schéma commun

In [ ]:
df = schema.valider(airbnb.transformer(airbnb.dedoublonner(brut)))
print(df.shape)
df.head()

In [ ]:
df.groupby(["ville", "tranche_dist"], observed=True).size().unstack()

## 5. Export intermédiaire

In [ ]:
INTERIM.mkdir(parents=True, exist_ok=True)
df.to_parquet(INTERIM / "airbnb.parquet", index=False)
print(INTERIM / "airbnb.parquet", len(df))